# FireGuard AI — BASS-II Data Exploration

Explores the 129-test BASS-II experimental table: shapes, oxygen measurements, fuel materials, and the derived O2-depletion proxy band.

**Note:** the BASS-II table has no flame-behavior label. The `o2_depletion_band` (low / moderate / high) is a tertile binning of measured O2 depletion derived by us — a burn-intensity proxy, not a NASA label.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import matplotlib.pyplot as plt
import pandas as pd

from data_processing.nasa_data_loader import load_experiments
from data_processing.data_cleaning import clean_experiments

df = clean_experiments(load_experiments("../data/experiments/bass2_experimental_table.csv"))
print(df.shape)
df.head()

In [ ]:
df[["test_id", "fuel_material", "initial_o2", "final_o2",
    "o2_depletion", "o2_depletion_band"]].describe(include="all")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["o2_depletion"].hist(bins=20, ax=axes[0])
axes[0].set_title("O2 depletion distribution (percentage points)")
axes[0].set_xlabel("initial O2 - final O2")
df["o2_depletion_band"].value_counts().plot.bar(ax=axes[1])
axes[1].set_title("Derived burn-intensity band counts")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for band, group in df.groupby("o2_depletion_band"):
    ax.scatter(group["initial_o2"], group["o2_depletion"], label=band, alpha=0.7)
ax.set_xlabel("Initial O2 % by vol")
ax.set_ylabel("O2 depletion (percentage points)")
ax.set_title("Initial O2 vs measured O2 depletion, colored by derived band")
ax.legend()
plt.show()

In [ ]:
print("Fuel materials:", sorted(df["fuel_material"].unique()))
print()
print(df.groupby("fuel_material")["o2_depletion"].agg(["count", "mean", "std"]))

## Takeaways

- 129 BASS-II tests; key measured drivers are initial/final O2, fuel material, and flow settings.
- O2 depletion (initial − final, both measured % by vol) is the observable the ML model learns to band.
- Next: run `python ml/train_model.py` to compare Random Forest vs Gradient Boosting, then explore predictions in the Streamlit app (`streamlit run app/main.py`).